# `lidar-rs` Federal Geospatial Data Download & Hydrology Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/lidar-rs/blob/main/examples/federal_data_download.ipynb)
[![PyPI](https://img.shields.io/pypi/v/lidar-rs.svg)](https://pypi.org/project/lidar-rs/)
[![crates.io](https://img.shields.io/crates/v/lidar.svg)](https://crates.io/crates/lidar)

Download authoritative open U.S. federal geospatial datasets for any specified region (WGS 84 bounding box, USGS HUC-2 through HUC-12 code, or local polygon vector file) in pure Rust with **zero GDAL dependencies and zero API keys**, then feed them straight into the `lidar-rs` dynamic Fill-Spill-Merge (FSM) hydrology pipeline:

1. **Query USGS 3DEP elevation products** (`query_3dep_products`) with newest-vintage tile deduplication (`1m`, `10m`, `30m`, or `lpc`)
2. **Download a complete federal data bundle** (`download_federal_data`): USGS WBD watersheds, USGS 3DEP DEM (mosaicked and clipped), USGS NHD flowlines, USFWS NWI wetlands, USGS National Transportation roads and railways, deterministic road-stream culverts, and NOAA Atlas 14 precipitation frequency tables
3. **Visualize all federal layers** (DEM, wetlands, streams, roads, synthesized culverts, and design-storm depths)
4. **Run the hydro-conditioned Fill-Spill-Merge pipeline** (`run_pipeline`) using the downloaded 3DEP DEM, synthesized culverts, NHD streams, and NOAA Atlas 14 100-yr 24-h design storm
5. **Call individual layer helpers** (`download_wbd`, `download_nhd`, `download_nwi`, `download_roads`, `synthesize_culverts`, `fetch_noaa_atlas14`) for custom workflows

In [ ]:
%pip install -q lidar-rs geopandas matplotlib pandas tifffile

In [ ]:
import json
import logging
from pathlib import Path

import geopandas as gpd
import lidar_rs as lidar
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile

logging.getLogger("tifffile").setLevel(logging.ERROR)
print(f"lidar-rs version: {lidar.__version__}")

## 1. Query Matching USGS 3DEP Tiles Without Downloading

Inspect available USGS 3DEP elevation tiles for a USGS HUC-12 watershed (`101600020102`, Headwaters Pipestem Creek in North Dakota) before downloading payloads. Overlapping historical vintages are automatically deduplicated to the newest `publication_date` per grid tile.

In [ ]:
HUC12 = "101600020102"  # Headwaters Pipestem Creek, North Dakota (~42 km^2)

tiles_1m = lidar.query_3dep_products(huc=HUC12, dem_product="1m")
df_1m = pd.DataFrame(tiles_1m)
df_1m["size_mb"] = (df_1m["size_bytes"].fillna(0) / 1e6).round(1)
print(f"1-m 3DEP tiles intersecting HUC-12 {HUC12}: {len(df_1m)} ({df_1m['size_mb'].sum():.1f} MB total)")
display(df_1m[["grid_key", "publication_date", "size_mb", "format", "title"]])

tiles_30m = lidar.query_3dep_products(huc=HUC12, dem_product="30m")
df_30m = pd.DataFrame(tiles_30m)
df_30m["size_mb"] = (df_30m["size_bytes"].fillna(0) / 1e6).round(1)
print(f"\n30-m (1 arc-second) 3DEP tiles: {len(df_30m)} ({df_30m['size_mb'].sum():.1f} MB total)")
display(df_30m[["grid_key", "publication_date", "size_mb", "format", "title"]])

## 2. Download Complete Federal Data Bundle for a Region

`download_federal_data` accepts **exactly one** region selector:
- `huc="101600020102"` (any USGS HUC-2 through HUC-12 code)
- `bbox=(min_lon, min_lat, max_lon, max_lat)` (WGS 84 decimal degrees)
- `region="study_area.geojson"` (any local `.geojson`, `.gpkg`, or `.shp` polygon)

Below we download all 7 federal layers for HUC-12 `101600020102` using `dem_product="30m"` (a single 48.8 MB tile) so Colab runs in seconds. Change `dem_product="1m"` to download, mosaic, and clip native 1-m USGS 3DEP LiDAR tiles instead.

In [ ]:
out_dir = Path("federal_huc12_demo")

bundle = lidar.download_federal_data(
    out_dir,
    huc=HUC12,
    datasets="all",          # wbd, dem, nhd, nwi, roads, culverts, noaa
    dem_product="30m",       # change to "1m" for native 1-m 3DEP LiDAR DEMs
    wbd_level=12,
    mosaic_dem=True,
    clip_to_region=True,
    tile_cache_dir=out_dir / "tiles_cache",
)

print("Region WGS 84 bbox:", bundle["region_bbox"])
print(f"WBD polygons:   {bundle['wbd_count']} -> {bundle['wbd_path']}")
print(f"3DEP DEM:       {len(bundle['dem_tiles'])} tile(s) -> {bundle['dem_mosaic_path']}")
print(f"NHD flowlines:  {bundle['nhd_count']} -> {bundle['nhd_path']}")
print(f"NWI wetlands:   {bundle['nwi_count']} -> {bundle['nwi_path']}")
print(f"USGS roads:     {bundle['roads_count']} -> {bundle['roads_path']}")
print(f"Culverts:       {bundle['culverts_count']} -> {bundle['culverts_path']}")
print(f"NOAA Atlas 14:  {bundle['noaa_atlas14_path']}")
print(f"Manifest:       {bundle['manifest_path']}")

## 3. Inspect NOAA Atlas 14 Design Storms & Plot Federal Layers

Inspect the partial-duration-series precipitation depth table (mm) and visualize the clipped DEM, USFWS NWI wetlands, USGS NHD flowlines, USGS National Transportation roads, and synthesized road-stream culverts.

In [ ]:
# NOAA Atlas 14 design storm depths (mm)
noaa = bundle["noaa_atlas14"]
df_noaa = pd.DataFrame(noaa["depths_mm"]).T
df_noaa.index.name = "Duration"
df_noaa.columns = [f"{ari}-yr" for ari in df_noaa.columns]
print(f"NOAA Atlas 14 PDS depths (mm) at ({noaa['lat']}, {noaa['lon']}):")
display(df_noaa.loc[["60-min", "6-hr", "12-hr", "24-hr", "2-day"]])

# Plot downloaded federal layers
wbd = gpd.read_file(bundle["wbd_path"])
nwi = gpd.read_file(bundle["nwi_path"])
nhd = gpd.read_file(bundle["nhd_path"])
roads = gpd.read_file(bundle["roads_path"])
culverts = gpd.read_file(bundle["culverts_path"]).to_crs(wbd.crs)

fig, ax = plt.subplots(figsize=(9, 7))
wbd.boundary.plot(ax=ax, color="black", linewidth=1.8, label="WBD HUC-12")
if len(nwi) > 0:
    nwi.plot(ax=ax, color="#3182bd", alpha=0.55, edgecolor="none", label=f"USFWS NWI ({len(nwi)})")
if len(roads) > 0:
    roads.plot(ax=ax, color="#636363", linewidth=0.8, alpha=0.75, label=f"USGS Roads ({len(roads)})")
if len(nhd) > 0:
    nhd.plot(ax=ax, color="#08519c", linewidth=1.3, label=f"USGS NHD ({len(nhd)})")
if len(culverts) > 0:
    culverts.geometry.centroid.plot(
        ax=ax, color="#de2d26", marker="^", markersize=65, edgecolor="white", linewidth=0.6,
        zorder=6, label=f"Synthesized Culverts ({len(culverts)})"
    )
ax.set_title(f"Authoritative Federal Layers for HUC-12 {HUC12}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="upper right", framealpha=0.92)
ax.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

## 4. Run Hydro-Conditioned Fill-Spill-Merge Pipeline on Downloaded Data

Project the clipped 30-m DEM to local UTM (`EPSG:26914`) in pure Rust using `lidar.reproject_image` (note: native `1m` 3DEP tiles are already in UTM meters and need no reprojection), align the synthesized culverts and NHD streams to the same UTM CRS, and run `lidar.run_pipeline` with dynamic Fill-Spill-Merge (`routing="fsm"`) forced by the downloaded NOAA Atlas 14 24-h 100-yr design storm.

In [ ]:
dem_utm = out_dir / "dem_utm_30m.tif"
lidar.reproject_image(
    bundle["dem_mosaic_path"],
    dem_utm,
    dst_crs="EPSG:26914",
    resampling="bilinear",
    resolution=(30.0, 30.0),
)

# Align streams and culverts to EPSG:26914 to match dem_utm
nhd_utm_path = out_dir / "nhd_utm.gpkg"
culverts_utm_path = out_dir / "culverts_utm.gpkg"
gpd.read_file(bundle["nhd_path"]).to_crs("EPSG:26914").to_file(nhd_utm_path, driver="GPKG")
gpd.read_file(bundle["culverts_path"]).to_crs("EPSG:26914").to_file(culverts_utm_path, driver="GPKG")

rain_100yr_mm = noaa["depths_mm"]["24-hr"][100]
print(f"Running dynamic FSM for 100-yr 24-h SCS Type II storm ({rain_100yr_mm:.1f} mm)...")

hyetograph = lidar.scs_type2_hyetograph(rain_100yr_mm, duration_h=24.0, time_step_h=1.0)
pipe_dir = out_dir / "fsm_100yr"
pipe = lidar.run_pipeline(
    dem_utm,
    pipe_dir,
    kernel_size=3,
    min_size=5,            # 5 x 900 m^2 = 0.45 ha at 30-m (use min_size=500 at 1-m)
    min_depth=0.2,
    interval=0.2,
    rain_intensity_cm_h=0.0,
    rain_duration_h=24.0,
    time_step_h=1.0,
    hyetograph=hyetograph,
    culverts_vector=culverts_utm_path,
    stream_vector=nhd_utm_path,
    routing="fsm",
    compress=True,
)

mb = pipe["fsm_mass_balance"]
ret_pct = 100.0 * mb["total_stored_m3"] / mb["total_runoff_m3"]
print(f"Nested depressions: {pipe['num_depressions']} (max level L{pipe['max_level']})")
print(f"Spill paths traced: {pipe['num_spill_paths']}")
print(f"100-yr storm storage retention: {ret_pct:.1f}% ({mb['total_stored_m3']/1e6:.2f} / {mb['total_runoff_m3']/1e6:.2f} Mm^3)")
print(f"Relative mass-balance error: {abs(mb['mass_balance_error_m3']) / mb['total_runoff_m3']:.2e}")

# Plot sink depth and dynamic fill time
sink = tifffile.imread(pipe_dir / "depth.tif")
ft = tifffile.imread(pipe_dir / "fill_time.tif")
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
im0 = axes[0].imshow(np.ma.masked_less_equal(sink, 0), cmap="Blues")
axes[0].set_title("Delineated Depression Depth (m)")
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], shrink=0.82, label="Depth (m)")

im1 = axes[1].imshow(np.ma.masked_less(ft, 0), cmap="viridis_r")
axes[1].set_title("Dynamic FSM Fill Time (hours, 100-yr Storm)")
axes[1].axis("off")
fig.colorbar(im1, ax=axes[1], shrink=0.82, label="Hours")
plt.tight_layout()
plt.show()

## 5. Individual Layer Helpers & Bounding Box / Polygon Downloads

You can also download any individual federal dataset by `bbox`, `huc`, or local polygon file (`region`), or synthesize culverts offline from any stream and road vector files:

In [ ]:
bbox_dir = Path("federal_bbox_demo")
bbox = (-99.14, 47.04, -99.08, 47.08)  # (min_lon, min_lat, max_lon, max_lat)

n_wbd = lidar.download_wbd(bbox_dir / "wbd.gpkg", bbox=bbox, level=12)
n_nhd = lidar.download_nhd(bbox_dir / "nhd.gpkg", bbox=bbox)
n_nwi = lidar.download_nwi(bbox_dir / "nwi.gpkg", bbox=bbox)
n_roads = lidar.download_roads(bbox_dir / "roads.gpkg", bbox=bbox)
n_culv = lidar.synthesize_culverts(
    bbox_dir / "nhd.gpkg",
    bbox_dir / "roads.gpkg",
    bbox_dir / "culverts.gpkg",
    half_length_m=25.0,
    snap_distance_m=15.0,
)
noaa_pt = lidar.fetch_noaa_atlas14(47.06, -99.11, out_json=bbox_dir / "noaa.json")

print(
    f"Bbox download complete: WBD={n_wbd}, NHD={n_nhd}, NWI={n_nwi}, "
    f"Roads={n_roads}, Culverts={n_culv}, 100-yr 24-h={noaa_pt['depths_mm']['24-hr'][100]:.1f} mm"
)